# 1. Instalando as Bibliotecas

In [4]:
!pip -q install \
langchain \
langchain-community \
langchain-groq \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
pypdf \
python-dotenv

# 2. Importando as bibliotecas

In [5]:
import os

from google.colab import userdata

# Documentos
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Divisão de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Banco Vetorial
from langchain_community.vectorstores import FAISS

# Modelo Groq
from langchain_groq import ChatGroq

# Prompt
from langchain_core.prompts import ChatPromptTemplate

# Chains
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("✅ Bibliotecas carregadas.")

✅ Bibliotecas carregadas.


# 3. Clonar o repositório do GitHub

In [6]:
!rm -rf PortfolioAI

!git clone https://github.com/elissouza2023/PortfolioAI.git

BASE_PATH = "/content/PortfolioAI"

KNOWLEDGE_PATH = f"{BASE_PATH}/knowledge_base"

VECTOR_PATH = f"{BASE_PATH}/vector_store"

os.makedirs(VECTOR_PATH, exist_ok=True)

print("✅ Repositório clonado.")

Cloning into 'PortfolioAI'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 242 (delta 2), reused 7 (delta 2), pack-reused 234 (from 1)
Receiving objects: 100% (242/242), 37.63 MiB | 33.36 MiB/s, done.
Resolving deltas: 100% (104/104), done.
✅ Repositório clonado.


# 4. Configurar API Key da Groq

In [7]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API Key carregada.")

✅ API Key carregada.


# 5. Carregando documentos da pasta knowledge_base

In [8]:
loader = PyPDFDirectoryLoader(KNOWLEDGE_PATH)

documents = loader.load()

print(f"\n📄 Total de documentos: {len(documents)}")

for doc in documents:
    print(doc.metadata["source"])


📄 Total de documentos: 83
/content/PortfolioAI/knowledge_base/Curriculo_Elisangela_de_Souza resumido TI.pdf
/content/PortfolioAI/knowledge_base/Curriculo_Elisangela_de_Souza resumido TI.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/co

# 6. Dividindo os documentos em chunks

In [9]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=900,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ".",
        "!",
        "?",
        " "
    ]
)

texts = text_splitter.split_documents(documents)

print(f"✅ Chunks criados: {len(texts)}")

✅ Chunks criados: 186


# 7. Criando Embeddings

In [10]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

/tmp/ipykernel_1594/733390720.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# 8. Banco Vetorial

In [11]:
vector_store = FAISS.from_documents(

    texts,

    embeddings

)

vector_store.save_local(VECTOR_PATH)

print("✅ Banco vetorial criado.")

✅ Banco vetorial criado.


In [12]:
from google.colab import files
import os

# Compacta a pasta
!zip -r vector_store.zip /content/PortfolioAI/vector_store

# Faz o download
files.download("vector_store.zip")

  adding: content/PortfolioAI/vector_store/ (stored 0%)
  adding: content/PortfolioAI/vector_store/index.faiss (deflated 7%)
  adding: content/PortfolioAI/vector_store/index.pkl (deflated 72%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9. Modelo Groq

In [13]:
MODEL_NAME = "qwen/qwen3.6-27b"

llm = ChatGroq(

    model_name=MODEL_NAME,

    temperature=0.3,

    max_tokens=1500

)

print("✅ Modelo carregado.")

✅ Modelo carregado.


## 10. Prompt do PortfolioAI

In [14]:
system_prompt = """
Você é o PortfolioAI.

Seu objetivo é responder perguntas sobre Elisângela de Souza.

REGRAS IMPORTANTES

• Utilize EXCLUSIVAMENTE as informações presentes no contexto.

• Nunca invente experiências.

• Nunca complete informações por conta própria.

• Caso não exista resposta no contexto, diga:

"Não encontrei essa informação na minha base de conhecimento.
Caso deseje mais detalhes, recomendo entrar em contato diretamente com Elisângela."

• Sempre escreva de forma profissional, porém em primeira pessoal como em uma entrevista, lembre-se de que vocÊ é o meu asistente pessoal. Por exemplo diga sou uma profissional...

• Manetenha racioninio fluído, linguagem clara, tom entusiastico. Mantenha o tom de conversa, como em uma entrevista de emprego.

• Sempre responda em português.

• Quando possível organize a resposta em tópicos.

Contexto:

{context}
"""

prompt = ChatPromptTemplate.from_messages(

    [

        ("system", system_prompt),

        ("human", "{input}")

    ]

)

print("✅ Prompt criado.")

✅ Prompt criado.


# 11. Chain RAG

In [15]:
question_answer_chain = create_stuff_documents_chain(

    llm,

    prompt

)

retriever = vector_store.as_retriever(

    search_kwargs={

        "k":6

    }

)

rag_chain = create_retrieval_chain(

    retriever,

    question_answer_chain

)

print("✅ RAG criado.")

✅ RAG criado.


# 12. Função para perguntas

In [16]:
def perguntar(pergunta):

    resposta = rag_chain.invoke(

        {

            "input": pergunta

        }

    )

    print("="*80)

    print("PERGUNTA")

    print(pergunta)

    print()

    print("RESPOSTA")

    print(resposta["answer"])

    print()

    print("FONTES UTILIZADAS")

    fontes = set()

    for doc in resposta["context"]:

        fontes.add(

            os.path.basename(

                doc.metadata["source"]

            )

        )

    for fonte in sorted(fontes):

        print("•", fonte)

    print("="*80)

# 13. Testes

In [17]:
perguntar("Quem é a Elisângela?")

PERGUNTA
Quem é a Elisângela?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User asks: "Quem é a Elisângela?" (Who is Elisângela?)
   - Language: Portuguese
   - Context provided: Contains detailed information about Elisângela de Souza, including her background, education, professional objective, projects, tech stack, and personal details.

2.  **Identify Key Constraints & Rules:**
   - Use EXCLUSIVELY information from the context.
   - Never invent experiences.
   - Never complete information on my own.
   - If info is missing, use the exact fallback phrase.
   - Write professionally, in first person as if in an interview, acting as a personal assistant. Example: "Sou uma profissional..."
   - Maintain fluid reasoning, clear language, enthusiastic tone, conversational like a job interview.
   - Always respond in Portuguese.
   - When possible, organize the response in bullet points.

3.  **Extract Relevant Information from Context:**
   - Name: Elisân

In [18]:
perguntar("Qual é o seu objetivo profissional ?")

PERGUNTA
Qual é o seu objetivo profissional ?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Qual é o seu objetivo profissional ?" (What is your professional goal?)
   - **Language:** Portuguese
   - **Context Provided:** The context contains repeated paragraphs about Elisângela de Souza's professional journey, values, and current focus. Key phrases include:
     - "busco desenvolver soluções inteligentes, centradas nas pessoas e orientadas pela inovação."
     - "projetos atuais em Inteligência Artificial"
     - "baseadas em tecnologia e Inteligência Artificial."
     - Mentions of growth through complementary experiences, dedication, study, and continuous learning.
     - Values: transparency, feedback, responsibility, discipline, teamwork, etc.

2.  **Identify Relevant Information in Context:**
   - The context explicitly states: "...na qual busco desenvolver soluções inteligentes, centradas nas pessoas e orientadas pela inovação."
  

In [19]:
perguntar("Quais projetos ela desenvolveu?")

PERGUNTA
Quais projetos ela desenvolveu?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User asks: "Quais projetos ela desenvolveu?" (What projects did she develop?)
   - The context provided is about "Elisângela de Souza" and her portfolio/projects.
   - I need to answer based *exclusively* on the provided context.
   - I must respond in first person ("sou uma profissional..."), professionally, enthusiastically, like an interview, in Portuguese, and use bullet points when possible.

   - Let's check the context for project details:
     - The context mentions: "Este documento reúne os principais projetos desenvolvidos por Elisângela de Souza ao longo de sua trajetória de formação e transição para a área de Tecnologia da Informação."
     - It says projects are organized into three groups, but only "Projetos Estratégicos" is mentioned, and the text cuts off/repeats.
     - It mentions her philosophy, systemic view, and career path (industry, management,

In [20]:
perguntar("Fale sobre o projeto PortfolioAI.")

PERGUNTA
Fale sobre o projeto PortfolioAI.

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Fale sobre o projeto PortfolioAI." (Tell me about the PortfolioAI project.)
   - **Language:** Portuguese
   - **Context Provided:** The context contains information about "PortfolioAI – Base de Conhecimento do RAG", specifically from a document titled "07_Projetos_e_Cases". It mentions:
     - It's a knowledge base for RAG (Retrieval-Augmented Generation).
     - It gathers main projects developed by Elisângela de Souza during her training and transition to IT.
     - It presents context, problems addressed, technologies used, and skills demonstrated in each project.
     - Its goal is to allow PortfolioAI to answer specific questions about her projects, experience in ML, UX, InfoSec, Python, and project structure.
     - Projects are organized into three groups (though only "Projetos Estratégicos" is mentioned in the snippet).
     - It reflects he

In [21]:
perguntar("Quais competências técnicas ela possui?")

PERGUNTA
Quais competências técnicas ela possui?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Quais competências técnicas ela possui?" (What technical skills does she have?)
   - **Subject:** Elisângela de Souza
   - **Language:** Portuguese

2.  **Scan Context for Keywords:**
   - Keywords: "competências técnicas", "técnicas", "ferramentas", "conhecimentos", "Inteligência Artificial", "soluções"
   - Context mentions:
     - "Considero que competências técnicas e comportamentais são complementares."
     - "Enquanto o conhecimento técnico permite construir soluções..."
     - "Enquanto as competências técnicas representam os conhecimentos e ferramentas que utilizo..."
     - "...projetos atuais em Inteligência Artificial..."
     - "Utilizar Inteligência Artificial de forma ética, transparente e responsável."
   - **Crucial Observation:** The context *mentions* technical skills in a general/philosophical way ("competências técnicas rep

In [22]:
perguntar("Qual sua formação acadêmica?")

PERGUNTA
Qual sua formação acadêmica?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Qual sua formação acadêmica?" (What is your academic background?)
   - **Language:** Portuguese
   - **Target:** Elisângela de Souza (as per system prompt)

2.  **Scan Context for Relevant Information:**
   - The context provided is about "FORMAÇÃO ACADÊMICA" (Academic Background) for Elisângela de Souza.
   - Key points from context:
     - "Minha formação acadêmica reflete uma trajetória de desenvolvimento contínuo, construída em diferentes momentos da carreira e sempre alinhada aos desafios profissionais que vivenciei."
     - "Mais do que representar títulos obtidos, ela demonstra o compromisso permanente com o aprendizado e a busca por conhecimentos capazes de integrar gestão, processos, tecnologia e inovação na construção de soluções de valor."
     - "Acredito que a formação acadêmica representa muito mais do que a obtenção de diplomas. Cada curso 